# Efficiency and Fairness per model

In [12]:
import sys
sys.path.append('../src')

import re
import ast
import pandas as pd

from Config.config import PATHS
from Utils.utils import GetMeasurements
from Utils.interaction import Performer
from Utils.indices import AlternationIndex
from Classes.cognitive_model_agents import *

In [2]:
path_to_data = PATHS['simulated_data'] / 'optimal_parameters.csv'
optimal_parameters = pd.read_csv(path_to_data)

In [3]:
def parse_params(s):
    if not isinstance(s, str):
        return s
    # Replace np.float64(x) and np.int64(x) with x
    s = re.sub(r'np\.float64\(([^)]+)\)', r'\1', s)
    s = re.sub(r'np\.int64\(([^)]+)\)', r'\1', s)
    return ast.literal_eval(s)

In [10]:
MODEL_BY_NAME = {m.name(): m for m in MODELS}

fixed_parameters = {
    'num_agents': 2,
    'threshold': 0.5,
}
simulation_parameters = {
    'num_rounds': 10,
    'num_episodes': 10,
    'verbose': False
}

df_list =[] 
for idx, row in optimal_parameters.iterrows():
    if row['model'].startswith('Priors'):
        continue
    model_class = MODEL_BY_NAME[row['model']]
    free_parameters = parse_params(row['params'])
    df = Performer.sim(
        agent_class=model_class,
        fixed_parameters=fixed_parameters,
        free_parameters=free_parameters,
        simulation_parameters=simulation_parameters
    )
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)
df.head(2)


,id_sim,round,attendance,id_player,decision,score,model,threshold,num_agents
0,1b8f33a6-232e-11f1-a48a-b5d0ad9dae1a,0,"[1, 0]",0,1,1,,0.5,2
1,1b8f33a6-232e-11f1-a48a-b5d0ad9dae1a,0,"[1, 0]",1,0,0,,0.5,2


In [18]:
gm = GetMeasurements(
    data=df, 
    measures=[
        'efficiency', 'inequality',
        'bounded_efficiency', 
        'entropy', 'conditional_entropy', 
        'conditional_entropy_2nd_order', 'min_entropy'
    ],
    normalize=True,
)
data = gm.get_measurements()
index_gen = AlternationIndex.from_file(priority='mlp')
data['inequality'] = index_gen(data)
data.head(2)

Exception: Error: No column data found. Should be one of "id_sim", "room", or "group".
Columns found: Index(['round', 'attendance', 'id_player', 'decision', 'score'], dtype='str')